In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/README.dataset.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/README.roboflow.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/data.yaml
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/valid/labels/005134_jpeg_jpg.rf.1472b7636b79a43e0cba96c5adf8f95f.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/valid/labels/006338_jpeg_jpg.rf.3e80a6eb806c15919d8a3723fca6eb0c.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/valid/labels/001681_jpeg_jpg.rf.89587ff8a4fec74706f1ebaa97c1b823.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/valid/labels/009935_jpeg_jpg.rf.2ec7b9279a26c03bafb6135144eb901a.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/valid/labels/005735_jpeg_jpg.rf.e2d5cbed103a9a5a748ff977917216b8.txt
/kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc/valid/labels

In [15]:
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm 
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import torchvision.transforms.functional as F
from torchvision.utils import draw_bounding_boxes

# Khởi tạo GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang chạy: {DEVICE}")

# 1. ĐƯỜNG DẪN KAGGLE (Nhớ thay 'tên-dataset-của-bạn' bằng tên thư mục thực tế trong mục Input)
# Thường nó sẽ có dạng: /kaggle/input/tên-dataset-của-bạn/data_chinhthuc
ROOT_DATA = '/kaggle/input/cpv-vehicle-detection-data/data_chinhthuc' 

# 2. ĐƯỜNG DẪN OUTPUT (Bắt buộc phải lưu vào /kaggle/working thì mới tải về được)
OUT_DIR = "/kaggle/working/retinanet_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

NUM_CLASSES = 6
CLASS_NAMES = ['Background', 'bus', 'construction', 'cyclist', 'pedestrian', 'vehicle']

# TĂNG THÔNG SỐ VÌ ĐÃ CÓ GPU MẠNH
MAX_SIDE = 480     # Trả lại kích thước lớn để model nhìn rõ xe nhỏ
BATCH_SIZE = 16     # Tăng batch size lên 8 hoặc 16
EPOCHS = 50
LR = 1e-5

Thiết bị đang chạy: cuda


In [16]:
class RetinaDatasetOptimized(Dataset):
    def __init__(self, root, split='train', max_side=256):
        self.img_dir = os.path.join(root, split, 'images')
        self.label_dir = os.path.join(root, split, 'labels')
        self.max_side = max_side
        
        if not os.path.exists(self.img_dir):
            raise FileNotFoundError(f"Không tìm thấy: {self.img_dir}")
        self.img_files = sorted([f for f in os.listdir(self.img_dir) if f.lower().endswith(('.jpg', '.png'))])

    def __len__(self): 
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        label_path = os.path.join(self.label_dir, self.img_files[idx].rsplit('.', 1)[0] + '.txt')
        
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        
        scale = min(self.max_side / max(W, H), 1.0)
        new_W, new_H = int(W * scale), int(H * scale)
        img = img.resize((new_W, new_H), Image.BILINEAR)
        
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) == 5:
                        c, x_c, y_c, wb, hb = map(float, parts)
                        x1 = (x_c - wb/2) * W * scale
                        y1 = (y_c - hb/2) * H * scale
                        x2 = (x_c + wb/2) * W * scale
                        y2 = (y_c + hb/2) * H * scale
                        if (x2 > x1) and (y2 > y1):
                            boxes.append([x1, y1, x2, y2])
                            labels.append(int(c) + 1)

        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64)
        }
        return transforms.ToTensor()(img), target

def collate_fn(batch): 
    return tuple(zip(*batch))

In [ ]:
# =========================================================
# TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN ROOT_DATA TRÊN KAGGLE
# =========================================================
kaggle_input_dir = '/kaggle/input'
auto_root_data = None

for root, dirs, files in os.walk(kaggle_input_dir):
    # Tìm nơi chứa thư mục 'train' và bên trong có 'images'
    if 'train' in dirs and os.path.exists(os.path.join(root, 'train', 'images')):
        auto_root_data = root
        break

if auto_root_data is None:
    raise FileNotFoundError("Không thể tìm thấy thư mục 'train/images' bên trong /kaggle/input. Hãy kiểm tra lại Dataset bạn đã upload.")
else:
    ROOT_DATA = auto_root_data
    print(f"Đã tự động nhận diện ROOT_DATA chuẩn: {ROOT_DATA}")
    
def get_model():
    model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
    num_anchors = model.head.classification_head.num_anchors
    model.head.classification_head = RetinaNetClassificationHead(256, num_anchors, NUM_CLASSES)
    return model.to(DEVICE)

model = get_model()
if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model) # Chia việc cho 2 card T4
model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_ds = RetinaDatasetOptimized(ROOT_DATA, 'train', MAX_SIDE)
val_ds = RetinaDatasetOptimized(ROOT_DATA, 'valid', MAX_SIDE)

# TẬN DỤNG Kaggle CPU: num_workers=2, pin_memory=True
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

history = {"train_loss": [], "val_loss": []}
best_loss = float('inf')
scaler = torch.amp.GradScaler('cuda', enabled=True)

print(f"training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False)
    
    for imgs, targets in train_loop:
        imgs = [img.to(DEVICE) for img in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        if all(len(t['boxes']) == 0 for t in targets): continue
        
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', enabled=True):
            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())
        
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        
        tr_loss += losses.item()
        train_loop.set_postfix(loss=losses.item())
    
    # Validation
    val_loss = 0
    model.train() 
    with torch.no_grad():
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Valid]", leave=False)
        for imgs, targets in val_loop:
            imgs = [img.to(DEVICE) for img in imgs]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            if all(len(t['boxes']) == 0 for t in targets): continue
            
            with torch.autocast(device_type='cuda', enabled=True):
                loss_dict = model(imgs, targets)
                val_loss += sum(loss for loss in loss_dict.values()).item()

    avg_tr = tr_loss / max(1, len(train_loader))
    avg_va = val_loss / max(1, len(val_loader))
    history["train_loss"].append(avg_tr)
    history["val_loss"].append(avg_va)
    scheduler.step()

    print(f"Epoch {epoch:02d}: Train {avg_tr:.4f} | Val {avg_va:.4f}")

    # LƯU MỖI 5 EPOCH
    if epoch % 5 == 0:
        torch.save(model.state_dict(), os.path.join(OUT_DIR, f"model_epoch_{epoch}.pth"))
    
    # Lưu Best Model
    if avg_va < best_loss:
        best_loss = avg_va        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_model.pth"))

    pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)

Đã tự động nhận diện ROOT_DATA chuẩn: /kaggle/input/datasets/duykhangf123/data-chinhthuc/data_chinhthuc
training...


Epoch 1/50 [Train]:   0%|          | 0/486 [00:00<?, ?it/s]

Epoch 1/50 [Valid]:   0%|          | 0/61 [00:00<?, ?it/s]

Epoch 01: Train 0.6587 | Val 0.5151


Epoch 2/50 [Train]:   0%|          | 0/486 [00:00<?, ?it/s]

...

Epoch 20: Train 0.2163 | Val 0.4065


Epoch 21/50 [Train]:   0%|          | 0/486 [00:00<?, ?it/s]

Epoch 21/50 [Valid]:   0%|          | 0/61 [00:00<?, ?it/s]

Epoch 21: Train 0.2121 | Val 0.4120


Epoch 22/50 [Train]:   0%|          | 0/486 [00:00<?, ?it/s]

In [ ]:
# 1. Vẽ đồ thị Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='s')
plt.title(f'RetinaNet Loss Curve (Subsampled {SUBSET_TRAIN_SIZE} imgs)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUT_DIR, 'loss_curve.png'))
plt.show()

# 2. Lưu Meta Data
meta = {
    "data_root": ROOT_DATA,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "lr": LR,
    "max_side": MAX_SIDE,
    "subset_train": SUBSET_TRAIN_SIZE,
    "model_name": "retinanet_resnet50_fpn_v2",
    "labels": {str(i): name for i, name in enumerate(CLASS_NAMES) if i > 0}
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

# 3. Tạo 5 ảnh Demo Predictions từ tập Valid
print("Đang tạo ảnh Demo Predictions...")
model.eval()

for i in range(min(5, len(val_ds))):
    img_tensor, target = val_ds[i]
    with torch.no_grad():
        preds = model([img_tensor.to(DEVICE)])[0]
    
    keep = preds['scores'] > 0.4
    boxes = preds['boxes'][keep]
    labels = preds['labels'][keep]
    
    img_uint8 = (img_tensor * 255).to(torch.uint8)
    pred_classes = [CLASS_NAMES[label.item()] for label in labels]
    
    if len(boxes) > 0:
        result_img = draw_bounding_boxes(img_uint8, boxes, labels=pred_classes, colors="red", width=2, font_size=10)
    else:
        result_img = img_uint8 
        
    result_pil = F.to_pil_image(result_img)
    result_pil.save(os.path.join(OUT_DIR, f"demo_pred_{i}.png"))

print(f" Kết quả tại: '{OUT_DIR}'")

In [ ]:
import json
import torchvision.transforms.functional as F
from torchvision.utils import draw_bounding_boxes
import matplotlib.pyplot as plt

# 1. Vẽ đồ thị Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss', marker='o', markersize=3)
plt.plot(history['val_loss'], label='Val Loss', marker='s', markersize=3)
# Thêm đường kẻ dọc vị trí Best Model
best_epoch = history['val_loss'].index(min(history['val_loss'])) + 1
plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Model (Epoch {best_epoch})')

plt.title('RetinaNet Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUT_DIR, 'loss_curve.png'))
plt.show()

# 2. Lưu Meta Data
meta = {
    "data_root": ROOT_DATA,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "lr": LR,
    "max_side": MAX_SIDE,
    "model_name": "retinanet_resnet50_fpn_v2",
    "labels": {str(i): name for i, name in enumerate(CLASS_NAMES) if i > 0}
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

# 3. Tạo 5 ảnh Demo Predictions từ tập Valid
print("\n📸 Đang tạo ảnh Demo Predictions...")
model.eval()

for i in range(min(5, len(val_ds))):
    img_tensor, target = val_ds[i]
    with torch.no_grad():
        preds = model([img_tensor.to(DEVICE)])[0]
    
    keep = preds['scores'] > 0.4
    boxes = preds['boxes'][keep]
    labels = preds['labels'][keep]
    
    img_uint8 = (img_tensor * 255).to(torch.uint8)
    pred_classes = [CLASS_NAMES[label.item()] for label in labels]
    
    if len(boxes) > 0:
        result_img = draw_bounding_boxes(img_uint8, boxes, labels=pred_classes, colors="red", width=2, font_size=10)
    else:
        result_img = img_uint8 
        
    result_pil = F.to_pil_image(result_img)
    result_pil.save(os.path.join(OUT_DIR, f"demo_pred_{i}.png"))

# Lưu thêm metrics vào file CSV để nộp báo cáo
metrics_df = pd.DataFrame([{
    'precision': precision, 'recall': recall, 'f1_score': f1_score,
    'accuracy': accuracy, 'mape_percent': mape
}])
metrics_df.to_csv(os.path.join(OUT_DIR, "evaluation_metrics.csv"), index=False)

print(f"✅ HOÀN TẤT! Toàn bộ file kết quả đã được lưu tại: '{OUT_DIR}'")
print("👉 Hãy mở thư mục /kaggle/working/retinanet_outputs ở cột bên phải để tải file về máy.")